# 01 — Scraping (Obtain, part 1)

**Goal**: build the raw dataset from scratch by scraping books.toscrape.com.

**Source**: [books.toscrape.com](https://books.toscrape.com) — a sandbox site explicitly built
for scraping practice. 1000 books across 50 categories, spread over 50 paginated listing pages.

**Why this source**: it's stable (it doesn't change, so this notebook won't silently break),
it's explicitly sanctioned for scraping, and it has a clean mix of numeric (price, rating,
stock, reviews) and categorical (category, availability) fields — exactly what's needed for
meaningful exploration later.

**Important limitation**: the site warns that prices and ratings are randomly assigned and
carry no real-world meaning. Any patterns found in later notebooks describe *this dataset*,
not the real book market. This is stated up front and revisited in the interpretation.

**Two-pass strategy**:
1. **Listing pages** (50 pages × 20 books) → title, price, star rating, availability, detail URL
2. **Detail pages** (1000 individual pages) → category, description, UPC, exact stock count,
   review count, tax breakdown

The second pass is necessary because *category* — the most useful categorical variable for
analysis — only exists in the breadcrumb on each book's own page.

**Design principle**: this notebook only *obtains*. No cleaning, no type conversion, no
filtering. Every field is stored exactly as scraped (hence the `_raw` suffixes). Cleaning is
`03_cleaning.ipynb`'s job. Keeping Obtain and Scrub separate means the expensive scrape runs
once, and cleaning logic can be rewritten freely without re-hitting the network.

In [ ]:
import sys
from pathlib import Path

# add project root (one level up from notebooks/) so "src" is importable
project_root = Path.cwd().parent
sys.path.append(str(project_root))
from src.scraper import scrape_all_books , scrape_all_details
import pandas as pd

In [3]:
books = scrape_all_books()  # no max_pages -> walks all pages until "next" disappears
len(books)

Scraped page 1 — 20 books (total so far: 20)
Scraped page 2 — 20 books (total so far: 40)
Scraped page 3 — 20 books (total so far: 60)
Scraped page 4 — 20 books (total so far: 80)
Scraped page 5 — 20 books (total so far: 100)
Scraped page 6 — 20 books (total so far: 120)
Scraped page 7 — 20 books (total so far: 140)
Scraped page 8 — 20 books (total so far: 160)
Scraped page 9 — 20 books (total so far: 180)
Scraped page 10 — 20 books (total so far: 200)
Scraped page 11 — 20 books (total so far: 220)
Scraped page 12 — 20 books (total so far: 240)
Scraped page 13 — 20 books (total so far: 260)
Scraped page 14 — 20 books (total so far: 280)
Scraped page 15 — 20 books (total so far: 300)
Scraped page 16 — 20 books (total so far: 320)
Scraped page 17 — 20 books (total so far: 340)
Scraped page 18 — 20 books (total so far: 360)
Scraped page 19 — 20 books (total so far: 380)
Scraped page 20 — 20 books (total so far: 400)
Scraped page 21 — 20 books (total so far: 420)
Scraped page 22 — 20 books

1000

## Pass 1 — listing pages

`scrape_all_books()` starts at the homepage and follows the "next" link until it disappears,
rather than hardcoding `range(1, 51)`. This is deliberate: it's self-terminating and would
still work if the catalogue grew or shrank.

In [4]:
df = pd.DataFrame(books)
print(df.shape)
print(df.dtypes)
df.head()
# df["title"].duplicated().sum()  # Check for duplicates in the "title" column

(1000, 5)
title               str
price_raw           str
availability_raw    str
star_rating_word    str
detail_url          str
dtype: object


,title,price_raw,availability_raw,star_rating_word,detail_url
0,A Light in the Attic,£51.77,In stock,Three,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,In stock,One,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,In stock,One,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,In stock,Four,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five,https://books.toscrape.com/catalogue/sapiens-a...


### Sanity check


1000 rows was expected (the site advertises 1000 results) and 1000 rows is what we got —
the pagination didn't skip or double-count any page.

One duplicate title appears. This is genuine site data (the same book listed twice), not a
scraping bug — the two rows have different detail URLs and UPCs. Kept as-is here; handled
explicitly in the cleaning notebook.

In [5]:
df.to_csv("../data/raw/books_raw.csv", index=False)
print("Saved:", df.shape)

Saved: (1000, 5)


## Pass 2 — detail pages

1000 requests at ~1.2s each ≈ 20 minutes. `scrape_all_details()` wraps each request in try/except so a single failure doesn't discard 20 minutes of work — failed URLs are recorded in a `scrape_error` column instead of raising.

In [6]:
details = scrape_all_details(df["detail_url"].tolist())
len(details)

Scraping detail pages:   0%|          | 0/1000 [00:00<?, ?it/s]

Scraping detail pages: 100%|██████████| 1000/1000 [19:56<00:00,  1.20s/it]


1000

In [7]:
details_df = pd.DataFrame(details)
print(details_df.shape)

if "scrape_error" in details_df.columns:
    failed = details_df["scrape_error"].notna().sum()
    print("Failed pages:", failed)
else:
    print("Failed pages: 0")

details_df.head()

(1000, 9)
Failed pages: 0


,detail_url,category,description,upc,price_excl_tax_raw,price_incl_tax_raw,tax_raw,availability_detail_raw,num_reviews_raw
0,https://books.toscrape.com/catalogue/a-light-i...,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,£51.77,£51.77,£0.00,In stock (22 available),0
1,https://books.toscrape.com/catalogue/tipping-t...,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,£53.74,£53.74,£0.00,In stock (20 available),0
2,https://books.toscrape.com/catalogue/soumissio...,Fiction,"Dans une France assez proche de la nôtre, un h...",6957f44c3847a760,£50.10,£50.10,£0.00,In stock (20 available),0
3,https://books.toscrape.com/catalogue/sharp-obj...,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,£47.82,£47.82,£0.00,In stock (20 available),0
4,https://books.toscrape.com/catalogue/sapiens-a...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,£54.23,£54.23,£0.00,In stock (20 available),0


In [8]:
books_full = df.merge(details_df, on="detail_url", how="left")
print(books_full.shape)
print(books_full.columns.tolist())
books_full.head()

(1000, 13)
['title', 'price_raw', 'availability_raw', 'star_rating_word', 'detail_url', 'category', 'description', 'upc', 'price_excl_tax_raw', 'price_incl_tax_raw', 'tax_raw', 'availability_detail_raw', 'num_reviews_raw']


,title,price_raw,availability_raw,star_rating_word,detail_url,category,description,upc,price_excl_tax_raw,price_incl_tax_raw,tax_raw,availability_detail_raw,num_reviews_raw
0,A Light in the Attic,£51.77,In stock,Three,https://books.toscrape.com/catalogue/a-light-i...,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,£51.77,£51.77,£0.00,In stock (22 available),0
1,Tipping the Velvet,£53.74,In stock,One,https://books.toscrape.com/catalogue/tipping-t...,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,£53.74,£53.74,£0.00,In stock (20 available),0
2,Soumission,£50.10,In stock,One,https://books.toscrape.com/catalogue/soumissio...,Fiction,"Dans une France assez proche de la nôtre, un h...",6957f44c3847a760,£50.10,£50.10,£0.00,In stock (20 available),0
3,Sharp Objects,£47.82,In stock,Four,https://books.toscrape.com/catalogue/sharp-obj...,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,£47.82,£47.82,£0.00,In stock (20 available),0
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five,https://books.toscrape.com/catalogue/sapiens-a...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,£54.23,£54.23,£0.00,In stock (20 available),0


## Output

`data/raw/books_raw.csv` — 1000 rows × 13 columns, listing and detail data merged on `detail_url`.

All values remain raw strings (`£51.77`, `"In stock (22 available)"`, `"Three"`).
Next: `02_api_enrichment.ipynb` adds real-world metadata from the Open Library API.

In [9]:
books_full.to_csv("../data/raw/books_raw.csv", index=False)
print("Saved:", books_full.shape)

Saved: (1000, 13)
